In [1]:
from pymilvus import MilvusClient


DEFAULT_MILVUS_URI = "http://localhost:19530"
client = MilvusClient(DEFAULT_MILVUS_URI)
client.list_databases()
client.use_database("dnd_agent")

In [17]:
# 创建 schema
schema = MilvusClient.create_schema()

# 自增主键字段
schema.add_field(field_name="spell_id", datatype=DataType.INT64, is_primary=True, auto_id=True)

schema.add_field(field_name="name", datatype=DataType.VARCHAR, max_length=128)
schema.add_field(field_name="level", datatype=DataType.INT64)
schema.add_field(field_name="school", datatype=DataType.VARCHAR, max_length=64)
schema.add_field(field_name="casting_time", datatype=DataType.VARCHAR, max_length=128)
schema.add_field(field_name="range", datatype=DataType.VARCHAR, max_length=128)

# 使用 ARRAY 类型存储 components 数组字段
schema.add_field(
    field_name="components",
    datatype=DataType.ARRAY,
    element_type=DataType.VARCHAR,
    max_capacity=3,
    max_length=32,
)

schema.add_field(field_name="duration", datatype=DataType.VARCHAR, max_length=128)
schema.add_field(field_name="description", datatype=DataType.VARCHAR, max_length=4096)

schema.add_field(field_name="ritual", datatype=DataType.BOOL)
schema.add_field(field_name="concentration", datatype=DataType.BOOL)

# 使用 ARRAY 类型存储 classes 数组字段
schema.add_field(
    field_name="classes",
    datatype=DataType.ARRAY,
    element_type=DataType.VARCHAR,
    max_capacity=16,
    max_length=256,
)

schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=128)

schema.add_field(field_name="description_vector", datatype=DataType.FLOAT_VECTOR, dim=1024)

print(schema)

{'auto_id': False, 'description': '', 'fields': [{'name': 'spell_id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'name', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}}, {'name': 'level', 'description': '', 'type': <DataType.INT64: 5>}, {'name': 'school', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 64}}, {'name': 'casting_time', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}}, {'name': 'range', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}}, {'name': 'components', 'description': '', 'type': <DataType.ARRAY: 22>, 'params': {'max_length': 32, 'max_capacity': 3}, 'element_type': <DataType.VARCHAR: 21>}, {'name': 'duration', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 128}}, {'name': 'description', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4

In [18]:
client.create_collection(collection_name="spells", schema=schema)

In [12]:
from pymilvus import DataType

client.describe_collection(collection_name="spells")

{'collection_name': 'spells',
 'auto_id': True,
 'num_shards': 1,
 'description': '',
 'fields': [{'field_id': 100,
   'name': 'spell_id',
   'description': '',
   'type': <DataType.INT64: 5>,
   'params': {},
   'auto_id': True,
   'is_primary': True},
  {'field_id': 101,
   'name': 'name',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 128}},
  {'field_id': 102,
   'name': 'level',
   'description': '',
   'type': <DataType.INT64: 5>,
   'params': {}},
  {'field_id': 103,
   'name': 'school',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 64}},
  {'field_id': 104,
   'name': 'casting_time',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 128}},
  {'field_id': 105,
   'name': 'range',
   'description': '',
   'type': <DataType.VARCHAR: 21>,
   'params': {'max_length': 128}},
  {'field_id': 106,
   'name': 'components',
   'description': '',
   'type': <DataType.ARRAY: 22>,


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embed_model = HuggingFaceEmbeddings(model_name="./models/bge-m3")

D:\workspace\py_projects\Familiar\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from load_spell_csv import load_spells_from_csv

spells = load_spells_from_csv("./data/dnd2024spell.csv")

In [6]:
from src.schemas.resources import Spell

def spell_to_milvus_data(spell: Spell) -> dict:
    return {
        "name": spell.name,
        "level": spell.level,
        "school": spell.school,
        "casting_time": spell.casting_time,
        "range": spell.range,
        "components": spell.components,
        "duration": spell.duration,
        "description": spell.description,
        "ritual": spell.ritual,
        "concentration": spell.concentration,
        "classes": spell.classes,
        "source": spell.source,
        "description_vector": embed_model.embed_query(f"{spell.name}: {spell.description}"),
    }

In [19]:
milvus_data = [spell_to_milvus_data(spell) for spell in spells]
client.insert(collection_name="spells", data=milvus_data)

{'insert_count': 537, 'ids': [458314914846755591, 458314914846755592, 458314914846755593, 458314914846755594, 458314914846755595, 458314914846755596, 458314914846755597, 458314914846755598, 458314914846755599, 458314914846755600, 458314914846755601, 458314914846755602, 458314914846755603, 458314914846755604, 458314914846755605, 458314914846755606, 458314914846755607, 458314914846755608, 458314914846755609, 458314914846755610, 458314914846755611, 458314914846755612, 458314914846755613, 458314914846755614, 458314914846755615, 458314914846755616, 458314914846755617, 458314914846755618, 458314914846755619, 458314914846755620, 458314914846755621, 458314914846755622, 458314914846755623, 458314914846755624, 458314914846755625, 458314914846755626, 458314914846755627, 458314914846755628, 458314914846755629, 458314914846755630, 458314914846755631, 458314914846755632, 458314914846755633, 458314914846755634, 458314914846755635, 458314914846755636, 458314914846755637, 458314914846755638, 4583149148

In [32]:
spell_index = client.prepare_index_params()
spell_index.add_index(field_name="name")
spell_index.add_index(field_name="level")
spell_index.add_index(field_name="school")
spell_index.add_index(field_name="casting_time")
spell_index.add_index(field_name="range")
spell_index.add_index(field_name="components")
spell_index.add_index(field_name="duration")
spell_index.add_index(field_name="description")
spell_index.add_index(field_name="ritual")
spell_index.add_index(field_name="concentration")
spell_index.add_index(field_name="classes")
spell_index.add_index(field_name="source")
spell_index.add_index(field_name="description_vector", index_type="IVF_FLAT", metric_type="COSINE")

client.create_index(collection_name="spells", index_params=spell_index)

In [27]:
client.load_collection(collection_name="spells")

In [30]:
res = client.query(collection_name="spells", filter="name == '火球术'", output_fields=["name", "description"])

In [38]:
res = client.query(collection_name="spells", filter="level >= 2 AND casting_time == '附赠动作' AND ARRAY_CONTAINS(classes, 'Wizard')", output_fields=["name", "level", "classes"])

In [39]:
res

data: ["{'name': '复苏秘法', 'level': 2, 'classes': ['Sorcerer', 'Wizard'], 'spell_id': 458314914846755693}", "{'name': '龙息术', 'level': 2, 'classes': ['Sorcerer', 'Wizard'], 'spell_id': 458314914846755707}", "{'name': '魔化武器', 'level': 2, 'classes': ['Paladin', 'Ranger', 'Sorcerer', 'Wizard'], 'spell_id': 458314914846755726}", "{'name': '迷踪步', 'level': 2, 'classes': ['Sorcerer', 'Warlock', 'Wizard'], 'spell_id': 458314914846755730}"]